In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Playground S6E7 — Student Health Risk (FE + SHAP selection + 5-learner stack)

A stacked ensemble for the 3-class, balanced-accuracy **student health risk** problem
(`unhealthy` / `at-risk` / `fit`), heavily imbalanced toward `at-risk`.

**Pipeline:**
1. **Feature engineering** — ratio/interaction features, nonlinear transforms, and a large
   family of *categorical variables derived from the numeric columns* (domain-threshold bins
   **and** train-fitted quantile bins), motivated directly by the companion EDA notebook.
2. **SHAP-based feature selection** — a fast XGBoost model is fit on the target-encoded
   feature matrix, TreeExplainer SHAP values are aggregated per original feature, and only the
   top contributors (by cumulative |SHAP|) are carried into the ensemble.
3. **Five base learners** → multinomial-LR meta-learner:
   **XGBoost + CatBoost + Logistic Regression + RealMLP + TabTransformer.**

**Base learners and why they're here.** The tree models (XGB, CatBoost) do the heavy lifting;
LR is a fast linear baseline; **RealMLP** (Holzmüller et al., NeurIPS 2024) and **TabTransformer**
(Huang et al., 2020) are neural learners included mainly for **diversity** — they make different
errors than the trees, which is what helps a stack even when a learner's standalone score is similar.

## What carries over unchanged
Multiclass OOF target encoding, 20%-sample Optuna HPO, balanced-accuracy optimization,
per-fold leakage discipline, the prior-correction toggle, and GPU auto-detection all carry over
from the earlier 4-learner version. The two new heavy pieces (SHAP selection, TabTransformer)
are documented in their own sections below.

## EDA facts this notebook relies on (from `s6e7-eda-baseline`)
- **Heavy target imbalance** — predicting `at-risk` for everything scores ~86% raw accuracy but
  only ~33% *balanced* accuracy, so every learner is class-balanced (weights or resampling).
- **Missingness is essentially random (MCAR)** — so imputation carries no signal; trees see raw
  NaNs natively, and the linear/neural branches median-impute only to satisfy their input contract.
- **No train/test distribution shift** — numeric and categorical distributions overlay almost
  perfectly, so transforms fitted on train (quantile edges, scalers, encoders) apply safely to test.
- **Most class-separable numerics:** `sleep_duration`, `bmi`, `exercise_duration`, `step_count` —
  these get the most derived features below.

## GPU support
All five learners use the GPU when one is available (**Settings → Accelerator → GPU**).
Detection is via `torch.cuda.is_available()`; each library gets its own flags. With no GPU it falls
back to CPU. RealMLP and TabTransformer are the branches that benefit most.


In [ ]:
!pip install -q optuna catboost pytabkit tab-transformer-pytorch shap

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils import resample
import xgboost as xgb
from catboost import CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier
import torch
import torch.nn as nn
from tab_transformer_pytorch import TabTransformer
import optuna, shap

warnings.filterwarnings("ignore"); optuna.logging.set_verbosity(optuna.logging.WARNING)
RNG, N_FOLDS, N_TRIALS = 42, 5, 25
TARGET = "health_condition"

# --- GPU configuration -------------------------------------------------------
# Set the Kaggle notebook Accelerator to "GPU" (Settings > Accelerator) to use
# this. Detected once via torch; each library gets its GPU flags below.
USE_GPU        = torch.cuda.is_available()
REALMLP_DEVICE = "cuda" if USE_GPU else "cpu"
print(f"GPU available: {USE_GPU}"
      + (f"  ({torch.cuda.get_device_name(0)})" if USE_GPU else ""))

def xgb_device_kwargs():
    # XGBoost >= 2.0: device='cuda' + tree_method='hist' (gpu_hist is deprecated)
    return {"tree_method": "hist", "device": "cuda" if USE_GPU else "cpu"}

def cat_device_kwargs():
    # CatBoost: task_type='GPU', devices='0' when a GPU is available
    return {"task_type": "GPU", "devices": "0"} if USE_GPU else {"task_type": "CPU"}

## 1. Load data and encode the target

Column-type detection is deferred until **after** feature engineering, so the new derived categoricals are routed correctly.

In [ ]:
df      = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
print("Train:", df.shape, " Test:", df_test.shape)

classes    = sorted(df[TARGET].unique())
cls_to_int = {c: i for i, c in enumerate(classes)}
int_to_cls = {i: c for c, i in cls_to_int.items()}
N_CLASSES  = len(classes)
print(f"Classes ({N_CLASSES}): {classes}")

y        = df[TARGET].map(cls_to_int).values
X        = df.drop(columns=["id", TARGET]).reset_index(drop=True)
X_test   = df_test.drop(columns=["id"]).reset_index(drop=True)
test_ids = df_test["id"].values

# raw schema (used by the feature-engineering block below)
RAW_NUM = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
           "step_count", "exercise_duration", "water_intake"]
RAW_CAT = ["diet_type", "stress_level", "sleep_quality",
           "physical_activity_level", "smoking_alcohol", "gender"]
RAW_NUM = [c for c in RAW_NUM if c in X.columns]
RAW_CAT = [c for c in RAW_CAT if c in X.columns]
print("Raw numeric:", RAW_NUM)
print("Raw categorical:", RAW_CAT)
print("Class balance:")
for c in classes: print(f"  {c}: {(df[TARGET] == c).mean():.3f}")

## 2. Feature engineering

Everything here is **unsupervised** (no target used), so it's leakage-safe to compute on train and
apply identically to test — and because the EDA showed *no train/test distribution shift*, quantile
edges fitted on train transfer cleanly to test.

Three families of features, plus a deliberately large set of derived categoricals so that the SHAP
selection step below has something meaningful to prune.

**(a) Ratio & interaction features (numeric).** The raw columns are individually informative, but
*combinations* often carry the physiological signal — e.g. calories burned *per minute of exercise*
(intensity), or heart rate *relative to* step count (a sedentary-but-elevated-HR signature). These
give the trees ready-made interactions they'd otherwise have to discover through deep splits:
`exercise_intensity`, `hydration_ratio`, `activity_ratio`, `cal_per_step`, `cal_per_kg_bmi`,
`steps_per_hr`, `water_per_cal`, `ex_per_step`, `metabolic_load`, `sedentary_index`,
`cardio_stress`, `activity_balance`, plus two composite scores (`fitness_score`, `sleep_hr_product`,
`ex_water_product`).

**(b) Nonlinear transforms (numeric).** `bmi_sq` captures the U-shaped health risk at both BMI
extremes; `log_steps` / `log_cal` tame right-skew; `sleep_dev` and `hr_dev` encode *distance from a
healthy set-point* (8 h sleep, ~70 bpm resting HR) rather than the raw level — a monotone feature is
easier for the linear/neural learners than a bump-shaped one.

**(c) Categorical variables *from* numeric columns (the requested transformation).** Two ways:
- **Domain-threshold bins** using clinical cut-points: `bmi_category` (under/normal/over/obese),
  `sleep_category`, `heart_rate_category`, `step_category`, `water_category`, `exercise_category`,
  `calorie_category`. These let the tree/neural learners treat "clinically overweight" as a discrete
  state, and — crucially — they feed the **multiclass target encoder**, which turns each bin into
  per-class rates (a smooth, powerful signal the raw float can't express directly).
- **Quantile bins** (`*_qbin`, quintiles) for the most separable numerics identified in the EDA
  (`sleep_duration`, `bmi`, `exercise_duration`, `step_count`, …). Edges are fit on **train only**
  and reused on test, so they're leakage-safe.

Binning the same numeric several different ways is intentionally redundant — the SHAP step keeps the
cuts that matter and drops the rest.


In [ ]:
EPS = 1e-3
def _safe(a, b): return a / (b + EPS)

def engineer_features(d):
    """Unsupervised feature engineering. Safe to apply independently to train and test."""
    d = d.copy()
    sleep, hr, bmi = d["sleep_duration"], d["heart_rate"], d["bmi"]
    cal,   steps   = d["calorie_expenditure"], d["step_count"]
    ex,    water   = d["exercise_duration"], d["water_intake"]

    # (a) ratios / interactions
    d["exercise_intensity"] = _safe(cal, ex)        # kcal per exercise-minute
    d["hydration_ratio"]    = _safe(water, steps)   # water per step
    d["activity_ratio"]     = _safe(ex, sleep)      # exercise vs sleep
    d["cal_per_step"]       = _safe(cal, steps)
    d["cal_per_kg_bmi"]     = _safe(cal, bmi)
    d["steps_per_hr"]       = _safe(steps, hr)
    d["water_per_cal"]      = _safe(water, cal)
    d["ex_per_step"]        = _safe(ex, steps)
    d["sleep_hr_product"]   = sleep * hr
    d["ex_water_product"]   = ex * water
    d["metabolic_load"]     = _safe(cal, sleep)     # burn per sleep-hour
    d["sedentary_index"]    = _safe(hr, steps)      # elevated HR, low movement
    d["cardio_stress"]      = _safe(hr, sleep)
    d["activity_balance"]   = _safe(ex, ex + sleep)
    d["fitness_score"]      = steps / 1000.0 + ex - _safe(bmi, 1.0) + sleep

    # (b) nonlinear transforms
    d["bmi_sq"]    = bmi ** 2
    d["log_steps"] = np.log1p(steps.clip(lower=0))
    d["log_cal"]   = np.log1p(cal.clip(lower=0))
    d["sleep_dev"] = (sleep - 8.0).abs()            # distance from ideal 8h
    d["hr_dev"]    = (hr - 70.0).abs()              # distance from ~70 bpm resting

    # (c1) domain-threshold categoricals from numeric (clinical cut-points)
    d["bmi_category"] = pd.cut(bmi, [-np.inf, 18.5, 25, 30, np.inf],
                               labels=["underweight", "normal", "overweight", "obese"]).astype("object")
    d["sleep_category"] = pd.cut(sleep, [-np.inf, 5, 8, np.inf],
                                 labels=["short", "normal", "long"]).astype("object")
    d["heart_rate_category"] = pd.cut(hr, [-np.inf, 60, 100, np.inf],
                                      labels=["low", "normal", "high"]).astype("object")
    d["step_category"] = pd.cut(steps, [-np.inf, 5000, 10000, np.inf],
                                labels=["low", "medium", "high"]).astype("object")
    d["water_category"] = pd.cut(water, [-np.inf, 1.5, 3, np.inf],
                                 labels=["low", "medium", "high"]).astype("object")
    d["exercise_category"] = pd.cut(ex, [-np.inf, 15, 45, np.inf],
                                    labels=["light", "moderate", "intense"]).astype("object")
    d["calorie_category"] = pd.cut(cal, [-np.inf, 1800, 2500, np.inf],
                                   labels=["low", "medium", "high"]).astype("object")
    return d

# (c2) quantile-bin categoricals — edges fit on TRAIN only, reused on test (leakage-safe)
QBIN_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
             "step_count", "exercise_duration", "water_intake",
             "fitness_score", "metabolic_load", "exercise_intensity"]

def fit_quantile_edges(d, cols, q=5):
    edges = {}
    for c in cols:
        e = np.unique(np.nanquantile(d[c].astype(float), np.linspace(0, 1, q + 1)))
        e[0], e[-1] = -np.inf, np.inf            # open the tails for unseen test values
        edges[c] = e
    return edges

def apply_quantile_bins(d, edges):
    d = d.copy()
    for c, e in edges.items():
        labs = [f"q{i}" for i in range(len(e) - 1)]
        d[f"{c}_qbin"] = pd.cut(d[c].astype(float), bins=e, labels=labs,
                                include_lowest=True).astype("object")
    return d

# apply FE
X      = engineer_features(X)
X_test = engineer_features(X_test)
qedges = fit_quantile_edges(X, QBIN_COLS, q=5)     # train-only edges
X      = apply_quantile_bins(X, qedges)
X_test = apply_quantile_bins(X_test, qedges)
print(f"After FE:  train {X.shape},  test {X_test.shape}")
print(f"Added {X.shape[1] - len(RAW_NUM) - len(RAW_CAT)} engineered columns")

### 2.1 Detect column types on the engineered frame

Same auto-detector as before, run *after* FE so the derived bins are treated as categorical (and later target-encoded) while ratios/transforms stay numeric. Categorical NaNs become an explicit `"nan"` string via the cast — missingness is a level, not a hole.

In [ ]:
def _is_stringy(s):
    if isinstance(s.dtype, pd.CategoricalDtype):   return True
    if s.dtype == object:                          return True
    if s.dtype.kind in ("O", "U", "S", "T"):       return True
    return pd.api.types.is_string_dtype(s) and not pd.api.types.is_numeric_dtype(s)

def detect_column_types(frame):
    cat = [c for c in frame.columns
           if _is_stringy(frame[c]) or (frame[c].dtype.kind in "iu" and frame[c].nunique() <= 12)]
    num = [c for c in frame.columns if c not in cat]
    return cat, num

CAT_COLS, NUM_COLS = detect_column_types(X)
for c in CAT_COLS:
    X[c] = X[c].astype(str); X_test[c] = X_test[c].astype(str)

print(f"Categorical ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numeric ({len(NUM_COLS)}): {NUM_COLS}")

## 3. Multiclass OOF target encoding + balanced sample weights

Each categorical column expands into K encoded columns (per-class rate), OOF + smoothed to prevent leakage. `balanced_sample_weight` gives the tree/linear learners class balancing; RealMLP gets balancing via resampling instead (below).

In [ ]:
def oof_target_encode_multiclass(X_tr, y_tr, X_te, cat_cols, n_classes,
                                 n_splits=5, seed=RNG, m=20.0):
    X_tr_enc = X_tr.drop(columns=cat_cols).copy()
    X_te_enc = X_te.drop(columns=cat_cols).copy()
    priors = np.array([(y_tr == k).mean() for k in range(n_classes)])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for col in cat_cols:
        oof = np.tile(priors, (len(X_tr), 1)).astype(float)
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            sub = pd.DataFrame({col: X_tr[col].iloc[tr_idx].values})
            for k in range(n_classes):
                sub[f"y{k}"] = (y_tr[tr_idx] == k).astype(float)
            agg = sub.groupby(col).agg(["sum", "count"])
            for k in range(n_classes):
                s = agg[(f"y{k}", "sum")]; c = agg[(f"y{k}", "count")]
                enc = (s + priors[k] * m) / (c + m)
                oof[val_idx, k] = X_tr[col].iloc[val_idx].map(enc).fillna(priors[k]).values
        for k in range(n_classes):
            X_tr_enc[f"{col}__te{k}"] = oof[:, k]

        sub = pd.DataFrame({col: X_tr[col].values})
        for k in range(n_classes):
            sub[f"y{k}"] = (y_tr == k).astype(float)
        agg = sub.groupby(col).agg(["sum", "count"])
        for k in range(n_classes):
            s = agg[(f"y{k}", "sum")]; c = agg[(f"y{k}", "count")]
            enc = (s + priors[k] * m) / (c + m)
            X_te_enc[f"{col}__te{k}"] = X_te[col].map(enc).fillna(priors[k]).values
    return X_tr_enc, X_te_enc

def balanced_sample_weight(y_arr):
    return compute_sample_weight(class_weight="balanced", y=y_arr)

## 4. SHAP-based feature selection on XGBoost

Feature engineering above produced a lot of columns, many deliberately redundant. Rather than guess
which survive, we let an XGBoost model *tell us* via SHAP.

**Method.**
1. Take a stratified 75/25 split; **multiclass-target-encode** the categoricals (5-fold OOF on the
   75% slice, applied to the 25% slice) so every column is numeric and leakage-safe.
2. Fit a quick, class-balanced XGBoost.
3. Compute **TreeExplainer SHAP values** on the held-out 25%. For a 3-class model SHAP returns one
   attribution per class; we take **mean |SHAP| across both samples and classes** as a single
   importance per *encoded* column.
4. Each categorical expands into `K` target-encoded columns (`col__te0..te{K-1}`); we **sum their
   importances back onto the source feature**, so a categorical is judged as a whole rather than
   penalized for being split across columns.
5. Rank original features and keep the smallest set reaching **`SHAP_CUM` (95%)** of total
   importance, with a floor of `MIN_FEATURES`.

**Why this helps.** The neural learners (RealMLP, TabTransformer) and LR are sensitive to noise
columns; pruning to the SHAP-selected core reduces overfitting risk and — since RealMLP/TabTransformer
train inside all five outer folds — meaningfully cuts wall-clock time. XGBoost's own SHAP is a natural
selector here because the tree models are the backbone of the stack.

> Note: SHAP importance measures *contribution to this XGBoost model*, not ground-truth causal
> relevance. It's a pragmatic filter, not a causal claim.


In [ ]:
SHAP_CUM     = 0.95     # keep features up to this cumulative share of |SHAP|
MIN_FEATURES = 12       # never drop below this many original features

# 1) stratified split + OOF target encoding (leakage-safe)
X_sel_tr, X_sel_va, y_sel_tr, y_sel_va = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG)
X_sel_tr = X_sel_tr.reset_index(drop=True); X_sel_va = X_sel_va.reset_index(drop=True)
Xtr_e, Xva_e = oof_target_encode_multiclass(
    X_sel_tr, y_sel_tr, X_sel_va, CAT_COLS, N_CLASSES, n_splits=5)

# 2) fast class-balanced XGBoost
sel_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
    colsample_bytree=0.8, objective="multi:softprob", num_class=N_CLASSES,
    eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
    n_jobs=-1, verbosity=0)
sel_model.fit(Xtr_e, y_sel_tr, sample_weight=balanced_sample_weight(y_sel_tr))

# 3) SHAP values on held-out slice -> mean|SHAP| per encoded column
sv = np.asarray(shap.TreeExplainer(sel_model).shap_values(Xva_e))
if sv.ndim == 3 and sv.shape[0] == N_CLASSES:      # (K, n, f)
    enc_imp = np.abs(sv).mean(axis=(0, 1))
elif sv.ndim == 3 and sv.shape[-1] == N_CLASSES:   # (n, f, K)
    enc_imp = np.abs(sv).mean(axis=(0, 2))
else:                                              # (n, f)
    enc_imp = np.abs(sv).mean(axis=0)
enc_cols = list(Xva_e.columns)
assert len(enc_imp) == len(enc_cols)

# 4) fold target-encoded columns back onto their source categorical
import re
def _to_original(colname):
    m = re.match(r"^(.*)__te\d+$", colname)
    return m.group(1) if m else colname
orig_imp = {}
for col, v in zip(enc_cols, enc_imp):
    o = _to_original(col)
    orig_imp[o] = orig_imp.get(o, 0.0) + float(v)
imp_ser = pd.Series(orig_imp).sort_values(ascending=False)

# 5) cumulative-importance cutoff
cum = (imp_ser / imp_ser.sum()).cumsum()
k = max(int((cum < SHAP_CUM).sum()) + 1, MIN_FEATURES)
k = min(k, len(imp_ser))
selected_features = list(imp_ser.index[:k])

print(f"Ranked {len(imp_ser)} original features; keeping top {k} "
      f"(cumulative |SHAP| = {cum.iloc[k-1]:.3f})\n")
print("Top 20 by aggregated |SHAP|:")
for name, val in imp_ser.head(20).items():
    mark = "  keep" if name in selected_features else "  drop"
    print(f"  {name:<26} {val:8.4f}{mark}")
dropped = [c for c in imp_ser.index if c not in selected_features]
print(f"\nDropped {len(dropped)}: {dropped}")

# 6) restrict the whole pipeline to the selected features
X      = X[selected_features].reset_index(drop=True)
X_test = X_test[selected_features].reset_index(drop=True)
CAT_COLS = [c for c in CAT_COLS if c in selected_features]
NUM_COLS = [c for c in NUM_COLS if c in selected_features]
print(f"\nFinal feature set: {len(selected_features)}  "
      f"(CAT={len(CAT_COLS)}, NUM={len(NUM_COLS)})")

## 5. Optuna HPO on 20% sample — optimizing balanced accuracy

Runs on the **SHAP-selected** feature set. Only XGB / CatBoost / LR are tuned here; RealMLP and TabTransformer use fixed sensible defaults (per-fold HPO on deep models would be prohibitively slow inside the outer stack).

In [ ]:
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.20, stratify=y, random_state=RNG)
X_sample = X_sample.reset_index(drop=True); y_sample = np.asarray(y_sample)


def cv_score_xgb(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        Xt_e, Xv_e = oof_target_encode_multiclass(Xt, yt, Xv, CAT_COLS, N_CLASSES, n_splits=3)
        m = xgb.XGBClassifier(**p, objective="multi:softprob", num_class=N_CLASSES,
                              eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
                              n_jobs=-1, early_stopping_rounds=50, verbosity=0)
        m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_e, yv)], verbose=False)
        sc.append(balanced_accuracy_score(yv, m.predict(Xv_e)))
    return float(np.mean(sc))


def cv_score_cat(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        m = CatBoostClassifier(**p, loss_function="MultiClass", eval_metric="TotalF1",
                               auto_class_weights="Balanced", random_seed=RNG,
                               **cat_device_kwargs(), verbose=0, allow_writing_files=False)
        m.fit(Xt, yt, cat_features=CAT_COLS, eval_set=(Xv, yv),
              use_best_model=True, early_stopping_rounds=50)
        sc.append(balanced_accuracy_score(yv, m.predict(Xv).ravel().astype(int)))
    return float(np.mean(sc))


def cv_score_lr(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        Xt_e, Xv_e = oof_target_encode_multiclass(Xt, yt, Xv, CAT_COLS, N_CLASSES, n_splits=3)
        # LR can't handle NaNs (unlike trees): median-impute then scale, fit on train only.
        imp = SimpleImputer(strategy="median").fit(Xt_e)
        scaler = StandardScaler().fit(imp.transform(Xt_e))
        m = LogisticRegression(**p, class_weight="balanced", max_iter=2000, n_jobs=-1)
        m.fit(scaler.transform(imp.transform(Xt_e)), yt)
        sc.append(balanced_accuracy_score(yv, m.predict(scaler.transform(imp.transform(Xv_e)))))
    return float(np.mean(sc))


def obj_xgb(t):
    return cv_score_xgb({
        "n_estimators":     t.suggest_int("n_estimators", 200, 1000),
        "learning_rate":    t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        t.suggest_int("max_depth", 3, 10),
        "min_child_weight": t.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample":        t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        t.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":       t.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "gamma":            t.suggest_float("gamma", 1e-8, 5.0, log=True)})

def obj_cat(t):
    return cv_score_cat({
        "iterations":    t.suggest_int("iterations", 300, 1200),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth":         t.suggest_int("depth", 4, 10),
        "l2_leaf_reg":   t.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True)})

def obj_lr(t):
    return cv_score_lr({"C": t.suggest_float("C", 1e-3, 10.0, log=True),
                        "solver": "lbfgs", "random_state": RNG})


studies = {}
for name, obj, n in [("xgb", obj_xgb, N_TRIALS), ("cat", obj_cat, N_TRIALS), ("lr", obj_lr, 15)]:
    t0 = time.time()
    s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RNG))
    s.optimize(obj, n_trials=n, show_progress_bar=False)
    studies[name] = s
    print(f"  {name}: bal_acc={s.best_value:.5f}  ({time.time()-t0:.1f}s)")
best = {k: v.best_params for k, v in studies.items()}

## 6. Base-learner fit functions (incl. RealMLP + TabTransformer)

Each returns `(val_proba, test_proba)`, both shape `(n, K)`, with `yv` passed explicitly for early
stopping. `fit_xgb`, `fit_cat`, `fit_lr`, `fit_realmlp` are unchanged from the previous version.

### TabTransformer — the new base learner
**TabTransformer** (Huang, Khetan, Cvitkovic & Karnin, 2020) embeds each categorical column as a
token and passes the token sequence through a stack of Transformer encoder layers, so the model
learns *contextual* categorical embeddings (a category's representation depends on the other
categoricals in the row). Those attended embeddings are concatenated with the normalized continuous
features and fed to an MLP head. It's a genuinely different inductive bias from GBDTs — its value in
this stack is **error diversity**, not necessarily a higher standalone score.

Adapting it to this balanced-accuracy multiclass problem needed four things, mirroring how RealMLP
was handled:
1. **Categorical → integer index with an "unknown" slot.** Transformer embeddings are integer
   lookups, so each categorical is mapped to `0..n-1` on the *training* fold, and every column
   reserves one extra index for categories unseen in that fold (they appear in val/test). This keeps
   all indices in range and prevents an embedding-lookup crash.
2. **Continuous standardization + median imputation** on train-fold statistics only (the transformer,
   like any net, needs finite, scaled inputs; trees don't).
3. **Class imbalance via a weighted cross-entropy loss** (`weight = N / (K · count_k)`) — the
   neural-net analogue of the trees' balanced sample weights.
4. **Early stopping on validation *balanced* accuracy**, keeping the best epoch's weights — we
   optimize the competition metric directly rather than plain accuracy.

Outputs are class-index-ordered `0..K-1` by construction (labels are trained as-is), so unlike
RealMLP no column reordering is needed. Falls back to CPU automatically when no GPU is present.


In [ ]:
def fit_xgb(Xt, yt, Xv, Xte, yv):
    Xt_e, Xv_e  = oof_target_encode_multiclass(Xt, yt, Xv,  CAT_COLS, N_CLASSES, n_splits=5)
    _,    Xte_e = oof_target_encode_multiclass(Xt, yt, Xte, CAT_COLS, N_CLASSES, n_splits=5)
    m = xgb.XGBClassifier(**best["xgb"], objective="multi:softprob", num_class=N_CLASSES,
                          eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
                          n_jobs=-1, early_stopping_rounds=50, verbosity=0)
    m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_e, yv)], verbose=False)
    return m.predict_proba(Xv_e), m.predict_proba(Xte_e)


def fit_cat(Xt, yt, Xv, Xte, yv):
    m = CatBoostClassifier(**best["cat"], loss_function="MultiClass", eval_metric="TotalF1",
                           auto_class_weights="Balanced", random_seed=RNG,
                           **cat_device_kwargs(), verbose=0, allow_writing_files=False)
    m.fit(Xt, yt, cat_features=CAT_COLS, eval_set=(Xv, yv),
          use_best_model=True, early_stopping_rounds=50)
    return m.predict_proba(Xv), m.predict_proba(Xte)


def fit_lr(Xt, yt, Xv, Xte, yv):
    Xt_e, Xv_e  = oof_target_encode_multiclass(Xt, yt, Xv,  CAT_COLS, N_CLASSES, n_splits=5)
    _,    Xte_e = oof_target_encode_multiclass(Xt, yt, Xte, CAT_COLS, N_CLASSES, n_splits=5)
    imp = SimpleImputer(strategy="median").fit(Xt_e)
    scaler = StandardScaler().fit(imp.transform(Xt_e))
    m = LogisticRegression(**best["lr"], class_weight="balanced", max_iter=2000,
                           n_jobs=-1, random_state=RNG)
    m.fit(scaler.transform(imp.transform(Xt_e)), yt)
    return (m.predict_proba(scaler.transform(imp.transform(Xv_e))),
            m.predict_proba(scaler.transform(imp.transform(Xte_e))))


def _class_balanced_resample(Xt, yt, seed=RNG):
    idx_by_c = {k: np.where(yt == k)[0] for k in np.unique(yt)}
    maxn = max(len(v) for v in idx_by_c.values())
    parts = [resample(idx_by_c[k], replace=True, n_samples=maxn, random_state=seed)
             for k in idx_by_c]
    res_idx = np.concatenate(parts)
    np.random.RandomState(seed).shuffle(res_idx)
    return Xt.iloc[res_idx].reset_index(drop=True), yt[res_idx]


def fit_realmlp(Xt, yt, Xv, Xte, yv):
    Xt_r, Xv_r, Xte_r = Xt.copy(), Xv.copy(), Xte.copy()
    # RealMLP rejects NaNs in continuous columns: median-impute (train medians only).
    num_here = [c for c in NUM_COLS if c in Xt_r.columns]
    if num_here:
        med = Xt_r[num_here].median()
        Xt_r[num_here]  = Xt_r[num_here].fillna(med)
        Xv_r[num_here]  = Xv_r[num_here].fillna(med)
        Xte_r[num_here] = Xte_r[num_here].fillna(med)
    for c in CAT_COLS:
        Xt_r[c]  = Xt_r[c].astype("category")
        Xv_r[c]  = Xv_r[c].astype("category")
        Xte_r[c] = Xte_r[c].astype("category")
    Xt_bal, yt_bal = _class_balanced_resample(Xt_r, yt)
    clf = RealMLP_TD_Classifier(
        random_state=RNG, device=REALMLP_DEVICE, n_threads=4, verbosity=0,
        val_metric_name="class_error", n_epochs=120, use_ls=False)
    clf.fit(Xt_bal, yt_bal)
    order = np.argsort(clf.classes_)
    return clf.predict_proba(Xv_r)[:, order], clf.predict_proba(Xte_r)[:, order]

# ---------------------------------------------------------------------------
# TabTransformer base learner (Huang et al., 2020) — self-contained wrapper.
# ---------------------------------------------------------------------------
TT_PARAMS = dict(dim=32, depth=6, heads=8, attn_dropout=0.1, ff_dropout=0.1,
                 mlp_hidden_mults=(4, 2))
TT_EPOCHS, TT_PATIENCE, TT_BS, TT_LR, TT_WD = 120, 16, 512, 1e-3, 1e-5

def fit_tabtransformer(Xt, yt, Xv, Xte, yv):
    torch.manual_seed(RNG); np.random.seed(RNG)
    dev = torch.device(REALMLP_DEVICE)
    cat_here = [c for c in CAT_COLS if c in Xt.columns]
    num_here = [c for c in NUM_COLS if c in Xt.columns]

    # --- categoricals -> integer index (train mapping; unseen -> reserved slot) ---
    if cat_here:
        cat_maps, cat_sizes = {}, []
        for col in cat_here:
            vals = sorted(Xt[col].astype(str).unique())
            cat_maps[col] = {v: i for i, v in enumerate(vals)}
            cat_sizes.append(len(vals) + 1)               # +1 = unknown slot
        def enc_cat(dfx):
            out = np.zeros((len(dfx), len(cat_here)), dtype=np.int64)
            for j, col in enumerate(cat_here):
                unk = len(cat_maps[col])
                out[:, j] = dfx[col].astype(str).map(cat_maps[col]).fillna(unk).astype(np.int64).values
            return out
    else:                                                 # degenerate: one constant token
        cat_sizes = [1]
        def enc_cat(dfx): return np.zeros((len(dfx), 1), dtype=np.int64)

    # --- continuous: train-median impute + train-stats standardize ---
    if num_here:
        med = Xt[num_here].median()
        mu  = Xt[num_here].fillna(med).mean()
        sd  = Xt[num_here].fillna(med).std().replace(0, 1.0)
        def enc_num(dfx): return ((dfx[num_here].fillna(med) - mu) / sd).values.astype(np.float32)
        n_cont = len(num_here)
    else:
        def enc_num(dfx): return np.zeros((len(dfx), 0), dtype=np.float32)
        n_cont = 0

    def to_dev(c, n):
        return (torch.as_tensor(c, dtype=torch.long, device=dev),
                torch.as_tensor(n, dtype=torch.float32, device=dev))
    Xc_tr, Xn_tr = to_dev(enc_cat(Xt),  enc_num(Xt))
    Xc_va, Xn_va = to_dev(enc_cat(Xv),  enc_num(Xv))
    Xc_te, Xn_te = to_dev(enc_cat(Xte), enc_num(Xte))
    y_tr = torch.as_tensor(yt, dtype=torch.long, device=dev)

    model = TabTransformer(categories=tuple(cat_sizes), num_continuous=n_cont,
                           dim_out=N_CLASSES, mlp_act=nn.ReLU(), **TT_PARAMS).to(dev)

    # balanced cross-entropy (neural analogue of balanced sample weights)
    cnt = np.bincount(yt, minlength=N_CLASSES).astype(float)
    cw  = torch.as_tensor(len(yt) / (N_CLASSES * np.clip(cnt, 1, None)),
                          dtype=torch.float32, device=dev)
    crit = nn.CrossEntropyLoss(weight=cw)
    opt  = torch.optim.AdamW(model.parameters(), lr=TT_LR, weight_decay=TT_WD)

    n = len(yt); idx = np.arange(n)
    best_ba, best_state, since = -1.0, None, 0
    for _ in range(TT_EPOCHS):
        model.train(); np.random.shuffle(idx)
        for s in range(0, n, TT_BS):
            b = torch.as_tensor(idx[s:s + TT_BS], dtype=torch.long, device=dev)
            opt.zero_grad()
            loss = crit(model(Xc_tr[b], Xn_tr[b]), y_tr[b])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vp = model(Xc_va, Xn_va).softmax(1).cpu().numpy()
        ba = balanced_accuracy_score(yv, vp.argmax(1))
        if ba > best_ba:
            best_ba, since = ba, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= TT_PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        pv = model(Xc_va, Xn_va).softmax(1).cpu().numpy()
        pt = model(Xc_te, Xn_te).softmax(1).cpu().numpy()
    return pv, pt

## 7. Outer 5-fold stacking (5 base learners)

Meta feature matrix is now `5 × K` wide; all predictions are out-of-fold. **RealMLP and TabTransformer are the slow branches** — on CPU with full data, budget accordingly; on GPU both are much faster.

In [ ]:
BASE_LEARNERS = ["xgb", "cat", "lr", "realmlp", "tabtransformer"]
N_BASE = len(BASE_LEARNERS)
def _slot(i): return slice(i * N_CLASSES, (i + 1) * N_CLASSES)

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RNG)
oof_meta  = np.zeros((len(X), N_BASE * N_CLASSES))
test_meta = np.zeros((len(X_test), N_BASE * N_CLASSES))
fold_scores = {k: [] for k in BASE_LEARNERS}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    Xt = X.iloc[tr_idx].reset_index(drop=True); Xv = X.iloc[va_idx].reset_index(drop=True)
    yt, yv = y[tr_idx], y[va_idx]; t0 = time.time()

    p_xgb_v, p_xgb_t = fit_xgb    (Xt, yt, Xv, X_test, yv)
    p_cat_v, p_cat_t = fit_cat    (Xt, yt, Xv, X_test, yv)
    p_lr_v,  p_lr_t  = fit_lr     (Xt, yt, Xv, X_test, yv)
    p_rmp_v, p_rmp_t = fit_realmlp(Xt, yt, Xv, X_test, yv)
    p_tt_v,  p_tt_t  = fit_tabtransformer(Xt, yt, Xv, X_test, yv)

    for i, (pv, pt) in enumerate([(p_xgb_v, p_xgb_t), (p_cat_v, p_cat_t),
                                  (p_lr_v, p_lr_t), (p_rmp_v, p_rmp_t),
                                  (p_tt_v, p_tt_t)]):
        oof_meta[va_idx, _slot(i)] = pv
        test_meta[:, _slot(i)]    += pt / N_FOLDS

    for i, k in enumerate(BASE_LEARNERS):
        fold_scores[k].append(balanced_accuracy_score(yv, oof_meta[va_idx, _slot(i)].argmax(1)))
    print(f"  Fold {fold} ({time.time()-t0:.1f}s)  "
          + "  ".join(f"{k.upper()}={fold_scores[k][-1]:.5f}" for k in BASE_LEARNERS))

print("\nBase-learner OOF balanced accuracy:")
for i, k in enumerate(BASE_LEARNERS):
    print(f"  {k:>7}: {balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1)):.5f}")

## 8. Meta-learner + per-learner contribution

The meta-contribution readout (sum of mean |coef| over each learner's class columns) tells you whether RealMLP and TabTransformer are earning their slots. A near-zero contribution means the meta-learner found it redundant with the trees — that's fine, the stack ignores it gracefully. A meaningful contribution means it's adding signal the trees missed.

In [ ]:
meta_oof_proba  = np.zeros((len(X), N_CLASSES))
meta_test_proba = np.zeros((len(X_test), N_CLASSES))
meta_coefs = []
skf_meta = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RNG + 1)
for tr, va in skf_meta.split(oof_meta, y):
    meta = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000,
                              n_jobs=-1, random_state=RNG)
    meta.fit(oof_meta[tr], y[tr])
    meta_oof_proba[va] = meta.predict_proba(oof_meta[va])
    meta_test_proba   += meta.predict_proba(test_meta) / N_FOLDS
    meta_coefs.append(np.abs(meta.coef_).mean(axis=0))

meta_bal_acc = balanced_accuracy_score(y, meta_oof_proba.argmax(1))
print(f"  Stacked meta OOF balanced accuracy: {meta_bal_acc:.5f}")
mc = np.mean(meta_coefs, axis=0)
print("  Meta contribution by base learner:")
for i, k in enumerate(BASE_LEARNERS):
    print(f"    {k:>7}: {mc[_slot(i)].sum():.3f}")

priors = np.array([(y == k).mean() for k in range(N_CLASSES)])
adj_bal_acc = balanced_accuracy_score(y, (meta_oof_proba / priors).argmax(1))
use_prior_correction = adj_bal_acc > meta_bal_acc
print(f"  With prior correction: {adj_bal_acc:.5f}  -> using: {use_prior_correction}")

print("\n=== Summary ===")
for i, k in enumerate(BASE_LEARNERS):
    print(f"  {k:>7} OOF   : {balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1)):.5f}")
avg = np.mean([oof_meta[:, _slot(i)] for i in range(N_BASE)], axis=0)
print(f"  simple avg  : {balanced_accuracy_score(y, avg.argmax(1)):.5f}")
print(f"  stacked meta: {meta_bal_acc:.5f}")

## 9. Submission

In [ ]:
final_pred = (meta_test_proba / priors).argmax(1) if use_prior_correction else meta_test_proba.argmax(1)
sub = pd.DataFrame({"id": test_ids, TARGET: [int_to_cls[i] for i in final_pred]})
sub.to_csv("submission.csv", index=False)
print("Submission shape:", sub.shape); print(sub.head())
print("\nPredicted class balance:")
print(sub[TARGET].value_counts(normalize=True))